In [ ]:
from pathlib import Path
import torch, os

KAGGLE_WORKING         = Path('/kaggle/working')
SRC_DATASET            = '/kaggle/input/datasets/maituananh511/test-dataset-chart-vqa/vi_chart_dataset'
DST_DATASET            = str(KAGGLE_WORKING / 'vi_chart_dataset')
VIETNAMESE_DATA_PATH   = Path('/kaggle/input/datasets/maituananh511/data-vietnamese/Data Vietnamese')
VIETNAMESE_IMAGES_PATH = VIETNAMESE_DATA_PATH / 'images'
VIETNAMESE_JSONL_PATH  = VIETNAMESE_DATA_PATH / 'viet_chart_vqa.jsonl'

QWEN_DIR      = KAGGLE_WORKING / 'qwen2vl_model'
QWEN_REPO_ID  = 'Qwen/Qwen2-VL-2B-Instruct'
VINTERN_LORA_CSV = Path('/kaggle/input/datasets/maituananh511/700-vinternlora-result/debug_vintern_lora.csv')

CHART_TEST_N   = 500
VIETNAMESE_N   = 200
EVAL_TOTAL     = CHART_TEST_N + VIETNAMESE_N
BATCH_SIZE     = 8
MAX_NEW_TOKENS = 64
METRICS        = ['bleu', 'meteor', 'rouge1', 'rouge2', 'rougeL', 'bertscore']

n_gpu = torch.cuda.device_count()
print(f'GPU count: {n_gpu}')
for i in range(n_gpu):
    p = torch.cuda.get_device_properties(i)
    print(f'  GPU {i}: {p.name} — {p.total_memory // 1024**3} GB')
print(f'\nEval target : {EVAL_TOTAL} samples')
print(f'Qwen repo   : {QWEN_REPO_ID}')


In [ ]:
from huggingface_hub import snapshot_download
import shutil, os

if QWEN_DIR.exists() and not any(QWEN_DIR.iterdir()):
    shutil.rmtree(str(QWEN_DIR))
    print('Xoa folder rong')

if not QWEN_DIR.exists():
    QWEN_DIR.mkdir(parents=True, exist_ok=True)
    print(f'Downloading {QWEN_REPO_ID} ...')
    snapshot_download(
        repo_id=QWEN_REPO_ID,
        local_dir=str(QWEN_DIR),
        ignore_patterns=['*.msgpack', '*.h5', 'flax_model*', 'tf_model*'],
    )
    print('Download xong')
else:
    print(f'Model da co — {len(list(QWEN_DIR.iterdir()))} files')

print('Files:', os.listdir(str(QWEN_DIR))[:10])


In [ ]:
from datasets import load_from_disk
from PIL import Image
import json, shutil, os

def is_dataset_complete(dst, src):
    if not os.path.exists(dst):
        return False
    src_files = {os.path.relpath(os.path.join(r, f), src)
                 for r, _, fs in os.walk(src) for f in fs}
    dst_files = {os.path.relpath(os.path.join(r, f), dst)
                 for r, _, fs in os.walk(dst) for f in fs}
    missing = src_files - dst_files
    if missing:
        print(f'Thieu {len(missing)} files')
        return False
    return True

if is_dataset_complete(DST_DATASET, SRC_DATASET):
    print('Dataset da copy day du, skip.')
else:
    if os.path.exists(DST_DATASET):
        shutil.rmtree(DST_DATASET)
    print('Copying dataset ...')
    shutil.copytree(SRC_DATASET, DST_DATASET)
    print('Copy xong.')

vi_chart_dataset = load_from_disk(DST_DATASET)
print(vi_chart_dataset)

vietnamese_records = []
with open(VIETNAMESE_JSONL_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            vietnamese_records.append(json.loads(line))
print(f'Vietnamese records: {len(vietnamese_records)} loaded')


In [ ]:
from PIL import Image

def normalize_turn(turn):
    if isinstance(turn, str):
        return {'role': 'assistant', 'content': turn}
    if isinstance(turn, dict):
        role = turn.get('role') or turn.get('from', '')
        if role in ('human', 'user'):      role = 'user'
        elif role in ('gpt', 'assistant'): role = 'assistant'
        for rk in ('assistant', 'user', 'human', 'gpt'):
            if rk in turn and 'content' not in turn and 'role' not in turn:
                role = 'assistant' if rk in ('assistant', 'gpt') else 'user'
                return {'role': role, 'content': str(turn[rk])}
        return {'role': role, 'content': str(turn.get('content') or turn.get('value', ''))}
    return {'role': 'assistant', 'content': str(turn)}

chart_test_raw   = vi_chart_dataset['test']
chart_n          = min(CHART_TEST_N, len(chart_test_raw))
chart_test_items = [chart_test_raw[i] for i in range(chart_n)]
print(f'vi_chart test   : {chart_n} samples')

vn_test_items = []
for record in reversed(vietnamese_records):
    if len(vn_test_items) >= VIETNAMESE_N:
        break
    img_path = VIETNAMESE_IMAGES_PATH / record['image']
    try:
        image = Image.open(img_path).convert('RGB')
    except Exception as e:
        print(f'Warning: {img_path}: {e}')
        continue
    convs = [normalize_turn(t) for t in record['conversations']]
    pairs = [(convs[i], convs[i+1]) for i in range(0, len(convs)-1, 2)]
    for idx, (q, a) in enumerate(pairs):
        if len(vn_test_items) >= VIETNAMESE_N:
            break
        rid = record['id'] if len(pairs) == 1 else f"{record['id']}_q{idx}"
        vn_test_items.append({'id': rid, 'image': image, 'conversations': [q, a]})

print(f'vietnamese test  : {len(vn_test_items)} samples')
eval_dataset = chart_test_items + vn_test_items
print(f'Total            : {len(eval_dataset)} samples')

has_img = sum(1 for x in eval_dataset if x.get('image') is not None)
print(f'Samples co anh   : {has_img} / {len(eval_dataset)}')


In [ ]:
def evaluate_qwen25vl_1gpu(eval_dataset, qwen_dir, batch_size=1, max_new_tokens=32):
    import os, gc, torch, nltk
    import pandas as pd
    from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
    from qwen_vl_utils import process_vision_info
    from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
    from nltk.translate.meteor_score import meteor_score as nltk_meteor
    from rouge_score import rouge_scorer
    from underthesea import word_tokenize
    from tqdm import tqdm
    from PIL import Image

    device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

    nltk_path = '/usr/share/nltk_data'
    os.makedirs(nltk_path, exist_ok=True)
    nltk.data.path.append(nltk_path)
    for pkg in ['punkt', 'wordnet', 'omw-1.4']:
        nltk.download(pkg, download_dir=nltk_path, quiet=True)

    processor = AutoProcessor.from_pretrained(
        str(qwen_dir),
        trust_remote_code=True,
        min_pixels=128 * 28 * 28,
        max_pixels=256 * 28 * 28,
    )

    mdl = Qwen2VLForConditionalGeneration.from_pretrained(
        str(qwen_dir),
        torch_dtype=torch.bfloat16,
        device_map={'': device},
        trust_remote_code=True,
    ).eval()

    items = [i for i in eval_dataset
             if all(k in i for k in ['id', 'conversations'])]
    print(f'  Model loaded — {len(items)} samples')

    system_msg = 'Bạn là trợ lý AI thông minh, chuyên phân tích biểu đồ. Hãy trả lời câu hỏi về biểu đồ trong ảnh bằng tiếng Việt ngắn gọn và chính xác.'

    scorer   = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    smoothie = SmoothingFunction().method1
    results  = []

    def get_pil_image(item):
        img = item.get('image')
        if img is None:
            return None
        if isinstance(img, Image.Image):
            pil = img
        elif isinstance(img, dict) and 'bytes' in img:
            import io
            pil = Image.open(io.BytesIO(img['bytes'])).convert('RGB')
        elif isinstance(img, (str, os.PathLike)):
            pil = Image.open(img).convert('RGB')
        else:
            return None
        return pil

    def build_messages(it):
        question = str(it['conversations'][0]['content'])
        pil_img  = get_pil_image(it)
        content  = []
        if pil_img is not None:
            content.append({'type': 'image', 'image': pil_img})
        content.append({'type': 'text', 'text': f'Câu hỏi: {question}'})
        return [
            {'role': 'system', 'content': system_msg},
            {'role': 'user',   'content': content},
        ]

    for it in tqdm(items, desc='Evaluating'):
        try:
            messages = build_messages(it)
            text     = processor.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True
            )
            image_inputs, video_inputs = process_vision_info(messages)

            enc = processor(
                text=[text],
                images=image_inputs if image_inputs else None,
                videos=video_inputs if video_inputs else None,
                padding=True,
                return_tensors='pt',
            ).to(device)

            with torch.no_grad():
                out = mdl.generate(
                    **enc,
                    max_new_tokens=max_new_tokens,
                    do_sample=False,
                    pad_token_id=processor.tokenizer.pad_token_id
                        if processor.tokenizer.pad_token_id is not None
                        else processor.tokenizer.eos_token_id,
                )

            input_len = enc['input_ids'].shape[1]
            response  = processor.tokenizer.decode(
                out[0][input_len:], skip_special_tokens=True
            ).strip()

        except torch.cuda.OutOfMemoryError:
            print(f'  OOM sample {it["id"]} — fallback text-only')
            torch.cuda.empty_cache()
            gc.collect()
            question = str(it['conversations'][0]['content'])
            messages_text = [
                {'role': 'system', 'content': system_msg},
                {'role': 'user',   'content': f'Câu hỏi: {question}'},
            ]
            text = processor.apply_chat_template(
                messages_text, tokenize=False, add_generation_prompt=True
            )
            enc = processor(text=[text], return_tensors='pt').to(device)
            with torch.no_grad():
                out = mdl.generate(
                    **enc,
                    max_new_tokens=max_new_tokens,
                    do_sample=False,
                    pad_token_id=processor.tokenizer.eos_token_id,
                )
            input_len = enc['input_ids'].shape[1]
            response  = processor.tokenizer.decode(
                out[0][input_len:], skip_special_tokens=True
            ).strip()

        finally:
            torch.cuda.empty_cache()
            gc.collect()

        gt  = str(it['conversations'][1]['content'])
        ref = word_tokenize(gt, format='text').split()
        hyp = word_tokenize(response, format='text').split() if response else ['']

        bleu   = sentence_bleu([ref], hyp, smoothing_function=smoothie)
        meteor = float(nltk_meteor([ref], hyp))
        r      = scorer.score(gt, response)

        results.append({
            'id':           it['id'],
            'question':     str(it['conversations'][0]['content']),
            'ground_truth': gt,
            'response':     response,
            'bleu':         bleu,
            'meteor':       meteor,
            'rouge1':       r['rouge1'].fmeasure,
            'rouge2':       r['rouge2'].fmeasure,
            'rougeL':       r['rougeL'].fmeasure,
        })

    df = pd.DataFrame(results)

    print('\nComputing BERTScore ...')
    try:
        import bert_score as bs_lib
        _, _, F1 = bs_lib.score(
            df['response'].tolist(),
            df['ground_truth'].tolist(),
            lang='vi', verbose=False, rescale_with_baseline=False,
        )
        df['bertscore'] = F1.tolist()
    except Exception as e:
        print(f'BERTScore failed: {e}')
        df['bertscore'] = [0.0] * len(df)

    metrics = ['bleu', 'meteor', 'rouge1', 'rouge2', 'rougeL', 'bertscore']
    avg = {m: df[m].mean() for m in metrics}
    return df, avg

print('Worker function defined')

In [ ]:
print('=' * 60)
print(f'Evaluating Qwen2.5-VL-2B-Instruct  [{len(eval_dataset)} samples]')
print(f'batch_size={BATCH_SIZE}  max_new_tokens={MAX_NEW_TOKENS}')
print('=' * 60)
df_qwen, avg_qwen = evaluate_qwen25vl_1gpu(
    eval_dataset,
    qwen_dir=QWEN_DIR,
    batch_size=BATCH_SIZE,
    max_new_tokens=MAX_NEW_TOKENS,
)
print('\nKet qua Qwen2.5-VL-2B-Instruct:')
for k, v in avg_qwen.items():
    print(f'  {k:<12}: {v:.4f}')
csv_qwen = KAGGLE_WORKING / 'debug_qwen25vl_2b.csv'
df_qwen.to_csv(str(csv_qwen), index=False, encoding='utf-8')
print(f'\nSaved: {csv_qwen}')


In [ ]:
import pandas as pd

print(f'Loading Vintern-LoRA tu: {VINTERN_LORA_CSV}')
df_vintern = pd.read_csv(str(VINTERN_LORA_CSV))
df_vintern.columns = [c.strip() for c in df_vintern.columns]

col_map = {c.lower(): c for c in df_vintern.columns}

avg_vintern = {}
for m in METRICS:
    actual_col = col_map.get(m.lower())
    if actual_col:
        avg_vintern[m] = df_vintern[actual_col].mean()
    else:
        avg_vintern[m] = 0.0
        print(f'  Khong tim thay cot "{m}", gan = 0.0')

print(f'Vintern-LoRA — {len(df_vintern)} samples')
print('\nKet qua Vintern-LoRA:')
for k, v in avg_vintern.items():
    print(f'  {k:<12}: {v:.4f}')


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

all_results = {
    'Qwen2.5-VL-2B': (df_qwen,    avg_qwen),
    'Vintern-LoRA':   (df_vintern, avg_vintern),
}

rows = []
for mname, (_, avg) in all_results.items():
    row = {'Model': mname}
    for m in METRICS:
        row[m.upper()] = round(avg.get(m, 0.0), 4)
    rows.append(row)

summary_df = pd.DataFrame(rows).set_index('Model')

def highlight_best(s):
    return ['background-color: #d4edda; font-weight: bold'
            if v == s.max() else '' for v in s]

def highlight_worst(s):
    return ['background-color: #f8d7da'
            if v == s.min() else '' for v in s]

print(f'KET QUA ({EVAL_TOTAL} samples | Xanh = cao nhat | Do = thap nhat)\n')
display(summary_df.style.apply(highlight_best).apply(highlight_worst))

csv_summary = KAGGLE_WORKING / 'eval_qwen25vl_vs_vintern_lora.csv'
summary_df.reset_index().to_csv(str(csv_summary), index=False, encoding='utf-8')
print(f'Saved: {csv_summary}')

n      = len(all_results)
x      = np.arange(len(METRICS))
width  = 0.8 / n
colors = ['#8e44ad', '#d65f5f']   # tim cho Qwen, do cho Vintern-LoRA

fig, ax = plt.subplots(figsize=(13, 5))
for idx, (mname, (_, avg)) in enumerate(all_results.items()):
    offset = idx * width - (n - 1) * width / 2
    vals   = [avg.get(m, 0.0) for m in METRICS]
    bars   = ax.bar(x + offset, vals, width, label=mname, color=colors[idx])
    ax.bar_label(bars, fmt='%.3f', padding=2, fontsize=8, rotation=90)

ax.set_xticks(x)
ax.set_xticklabels([m.upper() for m in METRICS], fontsize=10)
ax.set_ylabel('Score')
ax.set_ylim(0, 1.2)
ax.set_title(f'Qwen2.5-VL-2B-Instruct vs Vintern-LoRA  ({EVAL_TOTAL} samples)', fontsize=12)
ax.legend(fontsize=10)
plt.tight_layout()

img_path = KAGGLE_WORKING / 'eval_qwen25vl_vs_vintern_lora.png'
plt.savefig(str(img_path), dpi=150)
plt.show()
print(f'Saved chart: {img_path}')


In [ ]:
import pandas as pd

df_q = df_qwen[['id', 'question', 'ground_truth', 'response'] + METRICS].copy()
df_v = df_vintern[['id'] + [m for m in METRICS if m in df_vintern.columns]].copy()

df_merged = df_q.merge(df_v, on='id', suffixes=('_qwen', '_vintern'))
print(f'Merged: {len(df_merged)} samples')

for m in METRICS:
    cq, cv = f'{m}_qwen', f'{m}_vintern'
    if cq in df_merged.columns and cv in df_merged.columns:
        df_merged[f'delta_{m}'] = df_merged[cq] - df_merged[cv]

print('\nTop 10 Qwen2.5-VL vuot troi (delta BERTScore):')
display(df_merged.nlargest(10, 'delta_bertscore')
        [['id', 'question', 'delta_bleu', 'delta_meteor', 'delta_bertscore']])

print('\nTop 10 Qwen2.5-VL kem hon (delta BERTScore):')
display(df_merged.nsmallest(10, 'delta_bertscore')
        [['id', 'question', 'delta_bleu', 'delta_meteor', 'delta_bertscore']])

merged_csv = KAGGLE_WORKING / 'debug_qwen25vl_vs_vintern_merged.csv'
df_merged.to_csv(str(merged_csv), index=False, encoding='utf-8')
print(f'Saved: {merged_csv}')


In [ ]:
import matplotlib.pyplot as plt

color_map = {'Qwen2.5-VL-2B': '#8e44ad', 'Vintern-LoRA': '#d65f5f'}

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, m in enumerate(METRICS):
    ax    = axes[i]
    col_q = f'{m}_qwen'
    col_v = f'{m}_vintern'
    if col_q in df_merged.columns:
        ax.hist(df_merged[col_q], bins=30, alpha=0.6,
                color=color_map['Qwen2.5-VL-2B'], label='Qwen2.5-VL-2B')
    if col_v in df_merged.columns:
        ax.hist(df_merged[col_v], bins=30, alpha=0.6,
                color=color_map['Vintern-LoRA'], label='Vintern-LoRA')
    ax.set_title(m.upper(), fontsize=11)
    ax.set_xlabel('Score')
    ax.set_ylabel('Count')
    ax.legend(fontsize=8)

plt.suptitle(
    f'Phan phoi score: Qwen2.5-VL-2B vs Vintern-LoRA  ({len(df_merged)} samples)',
    fontsize=13,
)
plt.tight_layout()

hist_path = KAGGLE_WORKING / 'histogram_qwen25vl_vs_vintern_lora.png'
plt.savefig(str(hist_path), dpi=150)
plt.show()
print(f'Saved: {hist_path}')
